<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_3_Model_Selection_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Predicting Credit Default: Part 4
## model selection and Multiclass

author: Brad Sheese

---

## Introduction: Picking the Winner

Up to this point, we have used logistic regression exclusively. But just as in our regression notebooks, there are many algorithms we can choose from—Decision Trees, random forests, Support Vector Machines (SVMs), and more.

This notebook is divided into two sections:
1.  part a: model competition: We will use cross-validation and information criteria (aic/bic) to pick the best model for our Credit Default data.
2.  part b: entering the multiclass world: What if we have more than two categories (e.g., predicting the species of an animal or the type of a wine)? We will introduce the wine dataset and the strategies models use to handle multiple classes.

### Learning objectives
By the end of this notebook, you will be able to:
1.  Compare multiple model types using cross-validation.
2.  Understand the trade-off between goodness of fit and model complexity using aic and bic.
3.  Explain the difference between one-vs-rest (ovr) and softmax (multinomial) multiclass strategies.
4.  Apply classification metrics to a balanced multiclass dataset.

## problem 1: Part A setup - The Credit Default Data

We'll start by loading our familiar Credit Default data one last time to compare different algorithms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data loaded and ready for competition!")

## problem 2: Model Competition via cross-validation

Just like in our Regression Part 2 notebook, we don't want to rely on a single train/test split. We will use 5-fold cross-validation to find the model that generalizes best to unseen data.

We will test four competitors:
1.  logistic regression (Our baseline)
2.  decision tree (Simple, interpretable rules)
3.  random forest (An ensemble of many trees)
4.  svm (rbf) (Captures complex non-linear boundaries)

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'), X_train_scaled),
    'Decision Tree (d=3)': (DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced'), X_train),
    'Random Forest': (RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, class_weight='balanced'), X_train),
    'SVM (RBF Kernel)': (SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'), X_train_scaled)
}

cv_results = {}
for name, (model, X_feat) in models.items():
    scores = cross_val_score(model, X_feat, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<20}: Mean Accuracy = {scores.mean():.3f} (+/- {scores.std():.3f})")

# Visualize results
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(cv_results.values(), tick_labels=cv_results.keys())
ax.set_title('Model Performance Comparison (5-Fold CV Accuracy)')
ax.set_ylabel('Accuracy')
plt.show()
ax.grid(True, alpha=0.3)
plt.tight_layout()ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## problem 3: Evaluating Complexity (aic and bic)

Recall from Regression: we don't just want the model with the highest score; we want the simplest model that does the job well. 

- aic (Akaike Information Criterion) and bic (Bayesian Information Criterion) penalize models for having too many parameters. 
- lower is better.

Note: Calculating likelihood for non-parametric models like Trees is complex, so we will focus our aic/bic analysis on the logistic regression model, varying the amount of information it receives.

In [ ]:
def calc_aic_bic(model, X, y):
    n = len(y)
    # Get the log-likelihood
    log_likelihood = model.predict_log_proba(X)[np.arange(len(y)), y].sum()
    # Number of parameters (coefficients + intercept)
    k = model.coef_.shape[1] + 1
    
    aic = 2*k - 2*log_likelihood
    bic = k * np.log(n) - 2*log_likelihood
    return aic, bic

# Compare full model vs a restricted model (only top 5 features)
full_lr = LogisticRegression(max_iter=1000, random_state=42)
full_lr.fit(X_train_scaled, y_train)

restricted_lr = LogisticRegression(max_iter=1000, random_state=42)
restricted_lr.fit(X_train_scaled[:, :5], y_train)

f_aic, f_bic = calc_aic_bic(full_lr, X_train_scaled, y_train)
r_aic, r_bic = calc_aic_bic(restricted_lr, X_train_scaled[:, :5], y_train)

print(f"Full Model (48 features):       AIC={f_aic:.1f}, BIC={f_bic:.1f}")
print(f"Restricted Model (5 features): AIC={r_aic:.1f}, BIC={r_bic:.1f}")

## problem 4: Part B setup - Entering the Multiclass World

Up to now, we've only dealt with Binary outcomes (0 or 1). But what if we have three categories? 

To explore this, we switch to the wine dataset. This data contains the results of a chemical analysis of wines grown in the same region in Italy but derived from three different cultivars (varieties).

target categories: class 0, class 1, and class 2.

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print("Wine Dataset Properties:")
print(f"Classes: {np.unique(y_wine)}")
print(f"Class Distribution: {np.bincount(y_wine)}")

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine)
scaler_w = StandardScaler()
X_w_train_s = scaler_w.fit_transform(X_w_train)
X_w_test_s = scaler_w.transform(X_w_test)

## problem 5: Multiclass Strategies (OvR vs. Softmax)

Standard algorithms like logistic regression are binary by nature. To handle K classes, they use one of two "hacks":

1.  one-vs-rest (ovr): The model trains K separate binary classifiers. (class 0 vs. Not 0, class 1 vs. Not 1, etc.) The winner is the one with the highest probability.
2.  softmax (multinomial): The model is modified to have K outputs that are forced to sum to 1.0. This is a single, more complex optimization problem.

Let's visualize the results of an OvR strategy.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
ovr_lr = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
ovr_lr.fit(X_w_train_s, y_w_train)

y_w_proba = ovr_lr.predict_proba(X_w_test_s)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(3):
    axes[i].hist(y_w_proba[y_w_test == i, i], bins=10, alpha=0.7, label='Target Class', color='blue')
    axes[i].hist(y_w_proba[y_w_test != i, i], bins=10, alpha=0.4, label='Other Classes', color='red')
    axes[i].set_title(f'Classifier {i}: Is it Wine {i}?')
    axes[i].set_xlabel('Probability Score')
    axes[i].legend()
plt.tight_layout()
plt.show()

### Look Closely:
Notice how in each of the three plots, the blue samples (correct class) are pushed toward 1.0 and the red samples (other classes) are pushed toward 0.0. This shows three binary models working together to solve a 3-class problem.

## conclusion
In this notebook, we've transitioned from choosing models for binary problems to understanding how models scale to multiple classes. 

We've seen that:
1. cross-validation is the gold standard for picking a "winner" algorithm.
2. aic/bic help us avoid overfitting by penalizing complexity.
3. multiclass problems can be solved by breaking them down into multiple binary problems (OvR).

In our final notebook, part 5, we will look at how to evaluate these multiclass models and visualize the decision boundaries that these different algorithms create.